# HarnessChandelier

*Harness the Chandelier — connect to the center of your conversation.*  

**Topic Drift Tracking for Long-Running Agent Conversations**

In real-world AI agent interactions, users naturally drift across multiple topics —
then return to what they originally wanted.

HarnessChandelier doesn't try to prevent drift.
It **tracks it** — and finds the topic the user kept coming back to.

Like a chandelier at the center of a hall, the dominant topic stays fixed
no matter how much the conversation moves around it.

Built on:
- **BERTopic** + cuML UMAP/HDBSCAN — GPU-accelerated topic extraction
- **cuGraph PageRank** — topic importance ranking
- **Temporal edge weighting** — time-aware topic transition graph
- **Multilingual model (100+ languages)** - "intfloat/multilingual-e5-base" by Microsoft
- lang="ko" applies Korean stopwords. English text in Korean conversations is also supported.

Built for **Harness Engineering** workflows where conversations drift across topics.

This example is exactly the same as the English version but the content was translated to Korean

In [1]:
from harness_chandelier import HarnessChandelier

messages = [
    # A - 여행 계획
    "내년 봄에 일본 여행을 계획하고 싶어요. 어디서부터 시작해야 할까요?",
    "도쿄를 먼저 가야 할까요, 교토를 먼저 가야 할까요? 총 10일 일정이에요.",
    "JR패스는 어떻게 구매하는 게 좋을까요? 살 만한 가치가 있나요?",
    
    # B - 코딩 (갑자기)
    "완전 다른 얘기인데요. Python 코드에서 TypeError가 나고 있어요. 도움받을 수 있을까요?",
    "오류 메시지가 'NoneType object is not subscriptable'이라고 나오는데, 무슨 뜻인가요?",
    "해결했어요. 감사합니다. 단순한 null 체크 문제였어요.",
    
    # C - 인생 고민 (또 갑자기)
    "개인적인 걸 여쭤봐도 될까요? 직장을 그만둘까 생각 중이에요.",
    "이 회사에 7년째 다니고 있는데 정체된 느낌이에요. 성장이 없어요.",
    "월급은 괜찮은데 행복하지 않아요. 불확실성을 감수하고 나가는 게 맞을까요?",
    "부양해야 할 가족이 있어요. 그래서 그냥 그만두기가 더 어렵네요.",
    
    # A - 다시 여행
    "죄송해요, 다시 일본 얘기로. 10일 일정에 예산을 얼마나 잡아야 할까요?",
    "오사카도 가볼 만한가요, 아니면 교토에 시간을 더 쓰는 게 나을까요?",
    "신칸센은요? 소문대로 그렇게 빠른가요?",
    
    # B - 또 코딩
    "새로운 코딩 문제예요. 웹사이트 스크래핑을 하려는데 자꾸 막혀요.",
    "Selenium이랑 BeautifulSoup 중에 뭘 써야 할까요?",
    "그 웹사이트가 JavaScript 렌더링을 써요. 그러면 달라지나요?",
    "Selenium으로 되긴 됐는데 엄청 느려요. 빠르게 할 방법 있을까요?",
    
    # C - 다시 인생 고민
    "직장 얘기로 돌아가서요. 오늘 LinkedIn에서 헤드헌터 메시지가 왔어요.",
    "새 자리는 연봉이 30% 높은데 스타트업이에요. 위험 대 보상, 어떻게 보세요?",
    "와이프는 가라고 하고, 부모님은 안정적인 곳에 있으라고 하시네요.",
    "저라면 어떻게 하실 것 같아요?",
    
    # A - 또또 여행
    "일본 다시요 - 4월 항공권이 싸게 나왔어요. 지금 예약해야 할까요?",
    "벚꽃 시즌이 4월 맞죠? 제 일정이랑 겹칠까요?",
    "도쿄 호텔이 너무 비싸네요. 에어비앤비를 써볼까요?",
    "도쿄에서 처음 가는 사람한테 어떤 동네가 좋을까요?",
    
    # C - 또 인생 고민
    "스타트업에 지원했어요. 방금 1차 면접 봤어요.",
    "5년 후 목표가 뭐냐고 묻더라고요. 뭐라고 해야 할지 몰랐어요.",
    "망한 것 같아요. 많이 실망스럽네요.",
    "그냥 있던 데 있는 게 나을 것 같기도 해요. 적어도 안정적이니까요.",
    
    # B - 코딩 ㅋㅋ
    "다른 코딩 질문이에요. React 배우려고 하는데 어디서 시작해야 할까요?",
    "클래스 컴포넌트를 먼저 배우는 게 나을까요, 아니면 바로 훅으로 가도 될까요?",
    "첫 컴포넌트 만들었는데 props가 제대로 안 넘어가요.",
    "아 해결했어요. 중괄호가 빠져 있었어요. 역시나.",
    
    # A - 또또또 여행
    "일본 항공권 예약했어요! 4월 3일부터 13일까지요.",
    "이제 숙소를 정해야 해요. 10박 저렴한 옵션 추천해주실 수 있어요?",
    "비싸더라도 료칸 경험은 해봐야 할까요?",
    "후지산도 보고 싶어요. 도쿄 당일치기로 가는 게 낫나요, 근처에 1박 하는 게 낫나요?",
    "식이 제한 관련해서요. 저 채식주의자인데, 일본에서 힘들까요?",
    
    # C - 마지막으로 인생 고민 (진짜)
    "스타트업 2차 면접 연락이 왔어요! 망한 게 아니었나봐요.",
    "기술 프레젠테이션을 해오래요. 긴장되네요.",
    "이 직장을 얻으면 제 인생이 통째로 바뀌는 거잖아요. 설레야 하나요, 두려워야 하나요?",
    "계속 왔다갔다 해요. 어떤 날은 하고 싶고, 어떤 날은 하기 싫고.",
    
    # B - 코딩
    "React 앱 거의 다 만들었는데 배포가 헷갈려요.",
    "초보한테는 Vercel이랑 Netlify 중에 어느 게 더 쉬울까요?",
    "배포했어요! 근데 환경변수가 운영 환경에서 안 먹어요.",
    
    # A - 다시 여행!!!!!
    "일본까지 한 달 남았어요! 아직 실제 일정을 못 짰네요.",
    "10일 일본 여행 일정을 하루하루 짜주실 수 있어요?",
    "1일차 도쿄, 2일차 도쿄, 3일차 교토... 이게 현실적인가요?",
    
    # C - 인생 고민 마지막 몇 마디
    "입사 제안 받았어요. 금요일까지 대답해야 해요.",
    "너무 무서워요. 인생에서 이렇게 큰 결정은 처음이에요.",
    "받아들이려고요. 잘 되길 빌어주세요.",
    
    # A - 진짜 마지막으로 다시 여행
    "일본까지 2주 남았어요! 너무 설레요. 막판 팁 있으실까요?",
    "여행자 보험 들어야 할까요? 일본은 그게 필요할까요?",
    "4월 일본 여행 짐 싸기 리스트요. 뭘 꼭 챙겨야 할까요?",
    "내일 출발이에요. 마지막 조언 한 마디 해주세요.",
]

In [2]:
from datetime import datetime

real_timestamps = [
    # A - 여행 계획 (1월 10일, 오전)
    datetime(2026, 1, 10, 9, 0, 5),    # 0: 일본 여행 아이디어
    datetime(2026, 1, 10, 9, 1, 30),   # 1: 도쿄 vs 교토
    datetime(2026, 1, 10, 9, 3, 45),   # 2: JR패스

    # B - 코딩 (갑자기, 2시간 후)
    datetime(2026, 1, 10, 11, 0, 0),   # 3: TypeError 발생
    datetime(2026, 1, 10, 11, 2, 15),  # 4: NoneType 오류
    datetime(2026, 1, 10, 11, 15, 0),  # 5: 해결

    # C - 인생 고민 (다음날)
    datetime(2026, 1, 11, 9, 30, 0),   # 6: 퇴직 고민
    datetime(2026, 1, 11, 9, 32, 0),   # 7: 같은 회사 7년
    datetime(2026, 1, 11, 9, 35, 0),   # 8: 행복하지 않음
    datetime(2026, 1, 11, 9, 40, 0),   # 9: 부양가족

    # A - 여행 (다시, 점심 후)
    datetime(2026, 1, 11, 13, 0, 0),   # 10: 일본 예산
    datetime(2026, 1, 11, 13, 3, 0),   # 11: 오사카
    datetime(2026, 1, 11, 13, 5, 0),   # 12: 신칸센

    # B - 코딩 (저녁)
    datetime(2026, 1, 11, 20, 0, 0),   # 13: 웹 스크래핑
    datetime(2026, 1, 11, 20, 3, 0),   # 14: Selenium vs BeautifulSoup
    datetime(2026, 1, 11, 20, 8, 0),   # 15: JS 렌더링
    datetime(2026, 1, 11, 20, 30, 0),  # 16: Selenium 너무 느림

    # C - 인생 고민 (다시, 다음날 아침)
    datetime(2026, 1, 12, 8, 0, 0),    # 17: LinkedIn 헤드헌터
    datetime(2026, 1, 12, 8, 3, 0),    # 18: 연봉 30% 인상 스타트업
    datetime(2026, 1, 12, 8, 6, 0),    # 19: 와이프 vs 부모님
    datetime(2026, 1, 12, 8, 10, 0),   # 20: 당신이라면 어떻게?

    # A - 여행 (다시, 일주일 후)
    datetime(2026, 1, 19, 10, 0, 0),   # 21: 저렴한 항공권 발견
    datetime(2026, 1, 19, 10, 2, 0),   # 22: 벚꽃 시즌
    datetime(2026, 1, 19, 10, 5, 0),   # 23: 호텔 vs 에어비앤비
    datetime(2026, 1, 19, 10, 8, 0),   # 24: 도쿄 동네 추천

    # C - 인생 고민 (며칠 후)
    datetime(2026, 1, 22, 14, 0, 0),   # 25: 1차 면접
    datetime(2026, 1, 22, 14, 5, 0),   # 26: 5년 후 목표
    datetime(2026, 1, 22, 14, 10, 0),  # 27: 망한 것 같음
    datetime(2026, 1, 22, 14, 15, 0),  # 28: 그냥 있을까

    # B - 코딩 (저녁)
    datetime(2026, 1, 22, 21, 0, 0),   # 29: React 입문
    datetime(2026, 1, 22, 21, 5, 0),   # 30: 훅 vs 클래스
    datetime(2026, 1, 22, 21, 20, 0),  # 31: props 안 넘어감
    datetime(2026, 1, 22, 21, 45, 0),  # 32: 해결, 중괄호 문제

    # A - 여행 (2월)
    datetime(2026, 2, 1, 9, 0, 0),     # 33: 항공권 예약 완료!
    datetime(2026, 2, 1, 9, 5, 0),     # 34: 10박 숙소
    datetime(2026, 2, 1, 9, 10, 0),    # 35: 료칸 경험
    datetime(2026, 2, 1, 9, 15, 0),    # 36: 후지산
    datetime(2026, 2, 1, 9, 20, 0),    # 37: 일본에서 채식

    # C - 인생 고민
    datetime(2026, 2, 5, 10, 0, 0),    # 38: 2차 면접!
    datetime(2026, 2, 5, 10, 5, 0),    # 39: 기술 프레젠테이션
    datetime(2026, 2, 5, 10, 10, 0),   # 40: 인생이 바뀐다
    datetime(2026, 2, 5, 10, 15, 0),   # 41: 왔다갔다

    # B - 코딩
    datetime(2026, 2, 10, 20, 0, 0),   # 42: React 배포
    datetime(2026, 2, 10, 20, 10, 0),  # 43: Vercel vs Netlify
    datetime(2026, 2, 10, 20, 45, 0),  # 44: 환경변수 오류

    # A - 여행 (3월)
    datetime(2026, 3, 1, 9, 0, 0),     # 45: 한 달 남았다!
    datetime(2026, 3, 1, 9, 5, 0),     # 46: 하루하루 일정
    datetime(2026, 3, 1, 9, 10, 0),    # 47: 도쿄→교토 현실적인가?

    # C - 인생 고민
    datetime(2026, 3, 15, 18, 0, 0),   # 48: 입사 제안 받음!
    datetime(2026, 3, 15, 18, 5, 0),   # 49: 금요일까지 대답
    datetime(2026, 3, 15, 18, 10, 0),  # 50: 받아들이기로 함

    # A - 여행 (4월, 막판)
    datetime(2026, 4, 1, 8, 0, 0),     # 51: 2주 남았다!
    datetime(2026, 4, 1, 8, 5, 0),     # 52: 여행자 보험
    datetime(2026, 4, 1, 8, 10, 0),    # 53: 짐 싸기 리스트
    datetime(2026, 4, 2, 20, 0, 0),    # 54: 내일 출발!
]

In [3]:
ranker = HarnessChandelier(
    lang="ko",
    weights={"delta_time": 0.2}
)

result = ranker.fit(messages, timestamps=real_timestamps)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
print(f"Main Topic: {result.main_topic}")
print(f"Main Topic Keywords: {result.main_topic_keywords}")  #
print()
print("=== PageRank (Topic Importance) ===")
print(result.pagerank)

Main Topic: 1
Main Topic Keywords: ['날', '마지막', '조언', '정체', '출발']

=== PageRank (Topic Importance) ===
   vertex  pagerank
0       1  0.167582
1       0  0.146485
2       5  0.117012
3       2  0.112262
4       3  0.105671
5       4  0.093497
6       7  0.091672
7       6  0.087433
8       8  0.078386


In [5]:
print("=== Topic -1 (Outlier) messages ===")
for i, (msg, topic) in enumerate(zip(messages, result.topic_labels)):
    if topic == -1:
        print(f"[{i:02d}] {msg}")

=== Topic -1 (Outlier) messages ===
[02] JR패스는 어떻게 구매하는 게 좋을까요? 살 만한 가치가 있나요?


In [6]:
# Final Summary
print("=== Dominant Topic Analysis ===")
main_topic_messages = [
    (i, msg) for i, (msg, topic) 
    in enumerate(zip(messages, result.topic_labels)) 
    if topic == result.main_topic
]

print(f"Main Topic: {result.main_topic} (PageRank: {result.pagerank.iloc[0]['pagerank']:.4f})")
print(f"Appears in {len(main_topic_messages)} out of {len(messages)} messages")
print()
print("Messages classified as Main Topic:")
for i, msg in main_topic_messages:
    print(f"  [{i:02d}] {msg[:70]}...")

=== Dominant Topic Analysis ===
Main Topic: 1 (PageRank: 0.1676)
Appears in 7 out of 55 messages

Messages classified as Main Topic:
  [07] 이 회사에 7년째 다니고 있는데 정체된 느낌이에요. 성장이 없어요....
  [20] 저라면 어떻게 하실 것 같아요?...
  [26] 5년 후 목표가 뭐냐고 묻더라고요. 뭐라고 해야 할지 몰랐어요....
  [28] 그냥 있던 데 있는 게 나을 것 같기도 해요. 적어도 안정적이니까요....
  [41] 계속 왔다갔다 해요. 어떤 날은 하고 싶고, 어떤 날은 하기 싫고....
  [50] 받아들이려고요. 잘 되길 빌어주세요....
  [54] 내일 출발이에요. 마지막 조언 한 마디 해주세요....


In [ ]:
from harness_chandelier.summary import summarize

# Korean summary — Grok
print("=== Topic Summary (xAI Grok) [KO] ===")
summary_ko = summarize(
    result.main_topic_messages,
    provider="grok",
    language="ko"
)
print(summary_ko)

=== Topic Summary (xAI Grok) [KO] ===


사용자는 7년째 다니는 회사에서 성장 정체를 느끼며 이직 여부와 안정성 사이에서 지속적으로 갈등하고 조언을 구하는 데 관심이 있습니다.
